# TSP QAOA VQE with Autograd Gradient (Google Colab)

Runs TSP QAOA optimization using TREV's autograd-based gradient.

Compares:
- **Autograd**: 1 forward + 1 backward pass (exact, fast)
- **Parameter-shift**: 2P circuit evaluations (exact, standard)
- **QAOA-param** vs **Full-param** optimization subspace

Both MPS (chain) and TR (ring) topologies.

## 1. Setup (Colab)

In [ ]:
!pip install -q qiskit qiskit-optimization torch numpy pandas matplotlib seaborn hashable_list ordered_set
!pip install -q git+https://github.com/keunjunpark/TREV@real_form_autograd

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', module='scipy.sparse')

import math, time, gc, itertools
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import Any, List

from qiskit.circuit.library import QAOAAnsatz
from qiskit import transpile as qk_transpile
from qiskit.transpiler import CouplingMap
from qiskit_optimization.applications import Tsp
from qiskit_optimization.converters import QuadraticProgramToQubo

from TREV.circuit import Circuit
from TREV.hamiltonian.hamiltonian import Hamiltonian
from TREV.measure.enums import MeasureMethod
from TREV.transpile import from_qiskit
from TREV.optimization.gradients.autograd_gradient import AutogradGradient
from TREV.optimization.gradients.batch_parameter_shift import BatchParameterShiftGradient
from TREV.optimization.optimization import minimize as trev_minimize
from TREV.optimization.optimizer import Optimizer
from TREV.measure.right_suffix_sampling import argmax_bitstring_tr_right_suffix

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BASIS_GATES = ['swap', 'rzz', 'rx', 'h']
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem_in_bytes / 1e9:.1f} GB')

## 2. Shared Helpers

In [ ]:
@dataclass
class RunResult:
    """Stores results from a single optimization run."""
    method: str           # 'mps' or 'tr'
    optim_type: str       # 'qaoa_param' or 'full_param'
    grad_type: str        # 'autograd' or 'param_shift'
    n_cities: int
    reps: int
    seed: int
    rank: int
    n_qubits: int
    n_trev_params: int
    n_qaoa_params: int
    exp_values: List[float] = field(default_factory=list)
    best_results: List[Any] = field(default_factory=list)
    iter_times: List[float] = field(default_factory=list)
    peak_gpu_mb: float = 0.0


def make_tsp_ising(n_cities, seed):
    tsp_inst = Tsp.create_random_instance(n_cities, seed=seed)
    qp = tsp_inst.to_quadratic_program()
    qubo = QuadraticProgramToQubo().convert(qp)
    qubitOp, offset = qubo.to_ising()
    return qubitOp, offset, tsp_inst


def normalize_ising(qubitOp):
    max_coeff = max(abs(float(c.real)) for c in qubitOp.coeffs)
    if max_coeff > 0:
        qubitOp = qubitOp / max_coeff
    return qubitOp, max_coeff


def build_trev_hamiltonian(qubitOp):
    pauli_strings, coefficients = [], []
    for elm in qubitOp:
        pauli_strings.append(str(elm.paulis[0][::-1]))
        coefficients.append(float(elm.coeffs[0].real))
    return Hamiltonian(len(pauli_strings[0]), pauli_strings, coefficients)


def get_distance_matrix(tsp_inst):
    G = tsp_inst.graph
    n = len(G.nodes)
    dist = np.zeros((n, n))
    for i, j, data in G.edges(data=True):
        w = data.get('weight', 1.0)
        dist[i][j] = w
        dist[j][i] = w
    return dist


def solve_tsp_brute(dist):
    n = dist.shape[0]
    best_cost, best_tour = float('inf'), None
    for perm in itertools.permutations(range(n)):
        cost = sum(dist[perm[i], perm[(i + 1) % n]] for i in range(n))
        if cost < best_cost:
            best_cost, best_tour = cost, list(perm)
    return best_cost, best_tour


def decode_tsp_bitstring(bits, n_cities, qubit_perm=None):
    if isinstance(bits, str):
        bits = [int(b) for b in bits]
    bits = list(bits)
    if qubit_perm is not None:
        inv_perm = [0] * len(qubit_perm)
        for i, p in enumerate(qubit_perm):
            inv_perm[p] = i
        bits = [bits[inv_perm[i]] for i in range(len(bits))]
    n = n_cities
    N = n * n
    if len(bits) < N:
        return False, None
    matrix = np.array(bits[:N]).reshape(n, n)
    if not (np.all(matrix.sum(axis=1) == 1) and np.all(matrix.sum(axis=0) == 1)):
        return False, None
    tour = [int(np.argmax(matrix[:, t])) for t in range(n)]
    return True, tour


def compute_tour_cost(tour, dist):
    n = len(tour)
    return sum(dist[tour[i]][tour[(i+1) % n]] for i in range(n))


def route_for_chain(optimized, N, seed_transpiler=42):
    cm = CouplingMap.from_line(N)
    return qk_transpile(optimized, coupling_map=cm, optimization_level=1,
                        basis_gates=BASIS_GATES, seed_transpiler=seed_transpiler)


def route_for_ring(optimized, N, seed_transpiler=42):
    cm = CouplingMap.from_ring(N)
    return qk_transpile(optimized, coupling_map=cm, optimization_level=1,
                        basis_gates=BASIS_GATES, seed_transpiler=seed_transpiler)


print('Helpers loaded.')

## 3. QAOA Parameter Mapping

In [ ]:
def build_qaoa_mapping(routed_qc, rank, device, fuse_zz_swap=True):
    """Build the linear mapping from QAOA params to TREV theta.

    Returns: (trev_circuit, theta_base, J_matrix, qaoa_param_names)
    """
    params = routed_qc.parameters
    sorted_params = sorted(params, key=lambda p: p.name)
    qaoa_param_names = [p.name for p in sorted_params]
    K = len(qaoa_param_names)

    zero_bind = {p: 0.0 for p in params}
    qc_zero = routed_qc.assign_parameters(zero_bind)
    trev_circuit, theta_base = from_qiskit(qc_zero, fuse_zz_swap=fuse_zz_swap,
                                            rank=rank, device=device)
    P = theta_base.shape[0]

    J = torch.zeros(P, K, dtype=theta_base.dtype)
    for i, param in enumerate(sorted_params):
        unit_bind = {p: 0.0 for p in params}
        unit_bind[param] = 1.0
        qc_unit = routed_qc.assign_parameters(unit_bind)
        _, theta_unit = from_qiskit(qc_unit, fuse_zz_swap=fuse_zz_swap,
                                     rank=rank, device=device)
        J[:, i] = (theta_unit - theta_base).cpu()

    return trev_circuit, theta_base.cpu(), J, qaoa_param_names


print('Mapping functions loaded.')

## 4. Optimization Functions

In [ ]:
def run_vqe_full_param(circuit, theta0, hamil, n_iters, lr, grad_type='autograd',
                       shots=1000, measure_method=MeasureMethod.EFFICIENT_CONTRACTION):
    """Full-parameter optimization with selectable gradient method."""
    if grad_type == 'autograd':
        grad = AutogradGradient(
            measure_method=MeasureMethod.EFFICIENT_CONTRACTION,
            dtype=torch.complex128,
        )
        bvm = 'argmax_tr_noinv_BE'
    else:
        grad = BatchParameterShiftGradient(
            shift=math.pi / 2, batch_size=None, shots=shots,
            measure_method=measure_method, depth=1,
        )
        bvm = 'argmax_tr_noinv_BE'

    opt = Optimizer(torch.optim.Adam, {'lr': lr})
    theta_init = theta0.clone().detach().requires_grad_(True)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    new_theta, exp_vals, best_results, iter_times = trev_minimize(
        circuit, theta_init, hamil, opt, grad, n_iters,
        best_value_method=bvm,
    )

    peak_gpu_mb = 0.0
    if torch.cuda.is_available():
        peak_gpu_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    if hasattr(grad, '_gpu_pool') and grad._gpu_pool is not None:
        grad._gpu_pool.shutdown()
        grad._gpu_pool = None

    processed = []
    for t in best_results:
        if isinstance(t, str):
            processed.append([int(b) for b in t])
        else:
            processed.append(t.tolist() if hasattr(t, 'tolist') else list(t))

    return (
        [float(v.real) if hasattr(v, 'real') else float(v) for v in exp_vals],
        processed,
        list(iter_times),
        peak_gpu_mb,
    )


def run_vqe_qaoa_param(circuit, theta_base, J, hamil, qaoa_x0, n_iters, lr,
                       grad_type='autograd', shots=1000,
                       measure_method=MeasureMethod.EFFICIENT_CONTRACTION):
    """QAOA-parameterized optimization with selectable gradient method."""
    device = circuit.device
    theta_base_dev = theta_base.to(device)
    J_dev = J.to(device)
    qaoa_params = qaoa_x0.clone().detach().to(device)

    if grad_type == 'autograd':
        grad_estimator = AutogradGradient(
            measure_method=MeasureMethod.EFFICIENT_CONTRACTION,
            dtype=torch.complex128,
        )
    else:
        grad_estimator = BatchParameterShiftGradient(
            shift=math.pi / 2, batch_size=None, shots=shots,
            measure_method=measure_method, depth=1,
        )

    qaoa_params.requires_grad_(True)
    adam = torch.optim.Adam([qaoa_params], lr=lr)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    exp_values, best_results, iter_times = [], [], []
    start = time.time()

    with torch.no_grad():
        for epoch in range(n_iters):
            it_time = time.time()
            adam.zero_grad()

            full_theta = theta_base_dev + J_dev @ qaoa_params
            full_grad = grad_estimator.run(full_theta.detach(), circuit, hamil)
            qaoa_params.grad = J_dev.T @ full_grad
            adam.step()

            if device == 'cuda':
                torch.cuda.synchronize()
            iter_times.append(time.time() - it_time)

            # Reuse cached exp value from autograd if available
            cached_ev = getattr(grad_estimator, 'last_exp_value', None)
            if cached_ev is not None:
                exp_values.append(float(cached_ev.real) if hasattr(cached_ev, 'real') else float(cached_ev))
            else:
                full_theta_eval = theta_base_dev + J_dev @ qaoa_params
                exp_value = circuit.get_expectation_value(full_theta_eval, hamil,
                                                          MeasureMethod.EFFICIENT_CONTRACTION)
                exp_values.append(float(exp_value.real) if hasattr(exp_value, 'real') else float(exp_value))

            # Reuse cached tensor from autograd if available
            cached_tensor = getattr(grad_estimator, 'last_tensor', None)
            if cached_tensor is not None:
                tensor = cached_tensor
            else:
                full_theta_eval = theta_base_dev + J_dev @ qaoa_params
                tensor = circuit.build_tensor(full_theta_eval)
            best_bs = argmax_bitstring_tr_right_suffix(tensor)
            best_results.append(best_bs)

            elapsed = time.time() - start
            eta = (elapsed / (epoch + 1)) * (n_iters - epoch - 1) if epoch > 0 else 0
            print(f'\r  [{epoch+1}/{n_iters}] loss={exp_values[-1]:+.6f} '
                  f'|grad|={full_grad.norm().item():.2e} '
                  f'ETA={time.strftime("%M:%S", time.gmtime(eta))}', end='', flush=True)

            if epoch % 10 == 0:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    if hasattr(grad_estimator, '_gpu_pool') and grad_estimator._gpu_pool is not None:
        grad_estimator._gpu_pool.shutdown()
        grad_estimator._gpu_pool = None

    print()

    peak_gpu_mb = 0.0
    if torch.cuda.is_available():
        peak_gpu_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    processed = []
    for t in best_results:
        if isinstance(t, str):
            processed.append([int(b) for b in t])
        else:
            processed.append(t.tolist() if hasattr(t, 'tolist') else list(t))

    return exp_values, processed, iter_times, peak_gpu_mb


print('Optimization functions loaded.')

## 5. Experiment Configuration

In [ ]:
# ── Experiment grid ──
NC_LIST   = [3]           # number of cities (3 = 9 qubits, 4 = 16 qubits)
REPS_LIST = [1, 2]        # QAOA depth (params = 2*reps)
SEED_LIST = [0]           # random seeds
RANK_LIST = [8]           # bond dimension
METHODS   = ['tr']        # 'mps' (chain) or 'tr' (ring)

# ── Gradient methods to compare ──
GRAD_TYPES = ['autograd', 'param_shift']  # remove one to skip

# ── Optimization ──
N_ITERS = 100             # iterations per run
LR      = 0.1             # autograd gives exact gradients, so higher LR works well
SHOTS   = 1000            # only used by param_shift
INIT_SCALE = 0.5          # initial QAOA param scale (larger to escape flat |+> region)

# ── What to run ──
RUN_QAOA_PARAM = True     # QAOA-parameterized optimization
RUN_FULL_PARAM = False    # full-parameter optimization (slower)

total_configs = len(NC_LIST) * len(REPS_LIST) * len(SEED_LIST) * len(RANK_LIST) * len(METHODS)
optim_types = int(RUN_FULL_PARAM) + int(RUN_QAOA_PARAM)
n_grad = len(GRAD_TYPES)
print(f'Grid: {total_configs} configs x {optim_types} optim types x {n_grad} grad methods = {total_configs * optim_types * n_grad} runs')

## 6. Run Experiments

In [ ]:
all_results = []
tsp_ground_truth = {}

run_idx = 0
total_runs = total_configs * optim_types * n_grad

for nc in NC_LIST:
    for reps in REPS_LIST:
        for seed in SEED_LIST:
            torch.manual_seed(seed)
            np.random.seed(seed)

            qubitOp, offset, tsp_inst = make_tsp_ising(nc, seed)
            qubitOp_n, scale = normalize_ising(qubitOp)
            N = qubitOp_n.num_qubits
            hamil = build_trev_hamiltonian(qubitOp_n)

            if (nc, seed) not in tsp_ground_truth:
                dist = get_distance_matrix(tsp_inst)
                opt_cost, opt_tour = solve_tsp_brute(dist)
                tsp_ground_truth[(nc, seed)] = (opt_cost, opt_tour, dist)
                print(f'TSP(nc={nc}, seed={seed}): opt_cost={opt_cost:.2f}, N={N} qubits')

            qaoa = QAOAAnsatz(qubitOp_n, reps=reps)
            optimized = qk_transpile(qaoa, optimization_level=3, basis_gates=BASIS_GATES)

            for rank in RANK_LIST:
                for method in METHODS:
                    if method == 'mps':
                        routed = route_for_chain(optimized, N, seed_transpiler=seed)
                        ps_measure = MeasureMethod.PERFECT_SAMPLING
                    else:
                        routed = route_for_ring(optimized, N, seed_transpiler=seed)
                        ps_measure = MeasureMethod.RIGHT_SUFFIX_SAMPLING

                    K = 2 * reps
                    gen_seed = seed * 10000 + nc * 100 + reps
                    gen = torch.Generator().manual_seed(gen_seed)
                    qaoa_x0 = INIT_SCALE * torch.randn(K, generator=gen)

                    for grad_type in GRAD_TYPES:
                        # ── QAOA-parameterized ──
                        if RUN_QAOA_PARAM:
                            run_idx += 1
                            print(f'\n[{run_idx}/{total_runs}] QAOA-param ({grad_type}) | {method} | '
                                  f'nc={nc} reps={reps} seed={seed} rank={rank}')

                            trev_circ, theta_base, J, pnames = build_qaoa_mapping(
                                routed, rank, DEVICE, fuse_zz_swap=True)
                            P = theta_base.shape[0]
                            print(f'  Mapping: {K} QAOA -> {P} TREV params')

                            ev, br, it, gpu = run_vqe_qaoa_param(
                                trev_circ, theta_base, J, hamil, qaoa_x0,
                                N_ITERS, LR, grad_type=grad_type,
                                shots=SHOTS, measure_method=ps_measure)

                            res = RunResult(
                                method=method, optim_type='qaoa_param',
                                grad_type=grad_type,
                                n_cities=nc, reps=reps, seed=seed, rank=rank,
                                n_qubits=N, n_trev_params=P, n_qaoa_params=K,
                                exp_values=ev, best_results=br, iter_times=it,
                                peak_gpu_mb=gpu,
                            )
                            res.qubit_perm = trev_circ.qubit_perm
                            all_results.append(res)
                            print(f'  GPU: {gpu:.0f} MB | final_loss={ev[-1]:.6f} | '
                                  f'med_iter={np.median(it[2:])*1000:.0f}ms')

                            del trev_circ, theta_base, J
                            if torch.cuda.is_available():
                                torch.cuda.empty_cache()
                            gc.collect()

                        # ── Full-parameter ──
                        if RUN_FULL_PARAM:
                            run_idx += 1
                            print(f'\n[{run_idx}/{total_runs}] Full-param ({grad_type}) | {method} | '
                                  f'nc={nc} reps={reps} seed={seed} rank={rank}')

                            sorted_params = sorted(routed.parameters, key=lambda p: p.name)
                            bind_dict = {p: float(qaoa_x0[i]) for i, p in enumerate(sorted_params)}
                            qc_bound = routed.assign_parameters(bind_dict)
                            trev_circ_fp, theta0_fp = from_qiskit(
                                qc_bound, fuse_zz_swap=True, rank=rank, device=DEVICE)
                            P_fp = theta0_fp.shape[0]
                            print(f'  Params: {P_fp} TREV params (independent)')

                            ev, br, it, gpu = run_vqe_full_param(
                                trev_circ_fp, theta0_fp, hamil,
                                N_ITERS, LR, grad_type=grad_type,
                                shots=SHOTS, measure_method=ps_measure)

                            res = RunResult(
                                method=method, optim_type='full_param',
                                grad_type=grad_type,
                                n_cities=nc, reps=reps, seed=seed, rank=rank,
                                n_qubits=N, n_trev_params=P_fp, n_qaoa_params=K,
                                exp_values=ev, best_results=br, iter_times=it,
                                peak_gpu_mb=gpu,
                            )
                            res.qubit_perm = trev_circ_fp.qubit_perm
                            all_results.append(res)
                            print(f'  GPU: {gpu:.0f} MB | final_loss={ev[-1]:.6f} | '
                                  f'med_iter={np.median(it[2:])*1000:.0f}ms')

                            del trev_circ_fp, theta0_fp
                            if torch.cuda.is_available():
                                torch.cuda.empty_cache()
                            gc.collect()

print(f'\nDone! {len(all_results)} runs completed.')

## 7. Build Results DataFrame

In [ ]:
records = []
for r in all_results:
    opt_cost, opt_tour, dist = tsp_ground_truth[(r.n_cities, r.seed)]
    qperm = getattr(r, 'qubit_perm', None)

    best_ratio = 0.0
    ratios = []
    for bits in r.best_results:
        feasible, tour = decode_tsp_bitstring(bits, r.n_cities, qperm)
        if feasible and tour is not None:
            cost = compute_tour_cost(tour, dist)
            ratio = opt_cost / cost if cost > 0 else 0.0
            best_ratio = max(best_ratio, ratio)
        ratios.append(best_ratio)

    label = f'{r.method}_{r.optim_type}_{r.grad_type}'
    records.append({
        'method': r.method,
        'optim_type': r.optim_type,
        'grad_type': r.grad_type,
        'label': label,
        'n_cities': r.n_cities,
        'reps': r.reps,
        'seed': r.seed,
        'rank': r.rank,
        'n_qubits': r.n_qubits,
        'n_trev_params': r.n_trev_params,
        'n_qaoa_params': r.n_qaoa_params,
        'final_exp_value': r.exp_values[-1] if r.exp_values else np.nan,
        'best_exp_value': min(r.exp_values) if r.exp_values else np.nan,
        'final_accuracy': ratios[-1] if ratios else 0.0,
        'best_accuracy': max(ratios) if ratios else 0.0,
        'n_feasible': sum(1 for b in r.best_results
                         if decode_tsp_bitstring(b, r.n_cities, qperm)[0]),
        'med_iter_time': np.median(r.iter_times[2:]) if len(r.iter_times) > 2 else np.nan,
        'peak_gpu_mb': r.peak_gpu_mb,
        'accuracy_curve': ratios,
        'exp_curve': r.exp_values,
    })

df = pd.DataFrame(records)
print(f'{len(df)} runs')
df[['label', 'n_cities', 'reps', 'rank', 'seed',
    'n_trev_params', 'n_qaoa_params',
    'final_accuracy', 'best_accuracy', 'n_feasible',
    'med_iter_time', 'peak_gpu_mb']].round(3)

## 8. Convergence Plots: Autograd vs Parameter-Shift

In [ ]:
COLORS = {
    'autograd':    'tab:red',
    'param_shift': 'tab:blue',
}
STYLES = {
    'qaoa_param': '-',
    'full_param': '--',
}

nc_vals = sorted(df['n_cities'].unique())
reps_vals = sorted(df['reps'].unique())

fig, axes = plt.subplots(len(nc_vals), len(reps_vals),
                         figsize=(5*len(reps_vals), 4*len(nc_vals)),
                         squeeze=False, sharex=True)
fig.suptitle('Expectation Value Convergence: Autograd vs Param-Shift', fontsize=14, y=1.02)

for i, nc in enumerate(nc_vals):
    for j, reps in enumerate(reps_vals):
        ax = axes[i][j]
        sub = df[(df['n_cities'] == nc) & (df['reps'] == reps)]
        for _, row in sub.iterrows():
            color = COLORS.get(row['grad_type'], 'gray')
            ls = STYLES.get(row['optim_type'], '-')
            label = f"{row['method']} {row['optim_type']} ({row['grad_type']})"
            ax.plot(row['exp_curve'], color=color, ls=ls, label=label, alpha=0.8)
        ax.set_title(f'nc={nc}, reps={reps}', fontsize=10)
        if i == len(nc_vals) - 1:
            ax.set_xlabel('Iteration')
        if j == 0:
            ax.set_ylabel('Exp. Value')
        if i == 0 and j == 0:
            ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## 9. Accuracy Convergence

In [ ]:
fig, axes = plt.subplots(len(nc_vals), len(reps_vals),
                         figsize=(5*len(reps_vals), 4*len(nc_vals)),
                         squeeze=False, sharex=True)
fig.suptitle('TSP Accuracy (Running-Best): Autograd vs Param-Shift', fontsize=14, y=1.02)

for i, nc in enumerate(nc_vals):
    for j, reps in enumerate(reps_vals):
        ax = axes[i][j]
        sub = df[(df['n_cities'] == nc) & (df['reps'] == reps)]
        for _, row in sub.iterrows():
            color = COLORS.get(row['grad_type'], 'gray')
            ls = STYLES.get(row['optim_type'], '-')
            label = f"{row['method']} {row['optim_type']} ({row['grad_type']})"
            ax.plot(row['accuracy_curve'], color=color, ls=ls, label=label, alpha=0.8)
        ax.axhline(1.0, color='green', ls=':', alpha=0.4, label='Optimal')
        ax.set_ylim(-0.05, 1.1)
        ax.set_title(f'nc={nc}, reps={reps}', fontsize=10)
        if i == len(nc_vals) - 1:
            ax.set_xlabel('Iteration')
        if j == 0:
            ax.set_ylabel('Performance Ratio')
        if i == 0 and j == 0:
            ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## 10. Timing Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
time_data = df.groupby('label')['med_iter_time'].mean().sort_values()
colors = [COLORS.get(l.split('_')[-1] if 'autograd' in l else 'param_shift', 'gray') for l in time_data.index]
time_data.plot.barh(ax=ax, color=colors, edgecolor='black')
ax.set_xlabel('Median Iteration Time (s)')
ax.set_title('Per-Iteration Cost: Autograd vs Parameter-Shift')
plt.tight_layout()
plt.show()

# Speedup summary
for reps in reps_vals:
    sub = df[df['reps'] == reps]
    for ot in sub['optim_type'].unique():
        sub2 = sub[sub['optim_type'] == ot]
        t_ad = sub2[sub2['grad_type'] == 'autograd']['med_iter_time'].mean()
        t_ps = sub2[sub2['grad_type'] == 'param_shift']['med_iter_time'].mean()
        if t_ad > 0 and t_ps > 0:
            print(f'reps={reps} {ot}: autograd {t_ad*1000:.0f}ms, param_shift {t_ps*1000:.0f}ms, speedup {t_ps/t_ad:.2f}x')

## 11. Summary Table

In [ ]:
summary_cols = ['label', 'n_cities', 'reps', 'rank',
                'n_trev_params', 'n_qaoa_params',
                'best_accuracy', 'n_feasible', 'med_iter_time', 'peak_gpu_mb']

summary_table = df[summary_cols].copy()
summary_table['feasibility_%'] = (summary_table['n_feasible'] / N_ITERS * 100).round(1)
summary_table = summary_table.round(3)

print('=== Autograd vs Parameter-Shift Comparison ===')
display(summary_table.style.background_gradient(
    subset=['best_accuracy'], cmap='RdYlGn', vmin=0, vmax=1
).background_gradient(
    subset=['feasibility_%'], cmap='RdYlGn', vmin=0, vmax=100
))

## 12. Save Results

In [ ]:
save_df = df.drop(columns=['accuracy_curve', 'exp_curve'])
save_df.to_csv('tsp_qaoa_autograd_results.csv', index=False)
print(f'Saved {len(save_df)} rows to tsp_qaoa_autograd_results.csv')

curves = {}
for _, row in df.iterrows():
    key = f"{row['label']}_{row['n_cities']}_{row['reps']}_{row['seed']}_{row['rank']}"
    curves[key] = {
        'exp_curve': row['exp_curve'],
        'accuracy_curve': row['accuracy_curve'],
    }
torch.save(curves, 'tsp_qaoa_autograd_curves.pt')
print(f'Saved {len(curves)} curves to tsp_qaoa_autograd_curves.pt')